# 📐 Capítulo 3 — Análisis de Algoritmos
## Notación Asintótica y Funciones de Crecimiento

**Estructuras de Datos y Algoritmos — Lic. en Sistemas**  
*Goodrich, Tamassia & Goldwasser — Capítulo 3*

| Campo | Detalle |
|---|---|
| ⏱️ Duración | 100 minutos de contenido activo |
| 📊 Nivel | Intermedio |
| 🔗 Prerequisito | Cap. 1–2 (Python básico, OOP) |
| 🎯 Objetivos | 5 objetivos de aprendizaje |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap03/01_NotacionAsintotica_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook serás capaz de:

1. **Describir** las siete funciones de crecimiento canónicas y ordenarlas de menor a mayor
2. **Definir** formalmente Big-O, Big-Ω y Big-Θ y aplicarlas a funciones concretas
3. **Aplicar** las reglas prácticas para simplificar expresiones de complejidad
4. **Analizar** la complejidad temporal de bucles simples, anidados y código secuencial
5. **Contrastar** el análisis experimental con el matemático, identificando ventajas y limitaciones de cada enfoque

## 📋 Tabla de Contenidos

| # | Sección | Tiempo | Referencia |
|---|---------|--------|------------|
| 1 | Análisis experimental — motivación y límites | 15 min | §3.1 |
| 2 | Las siete funciones de crecimiento | 20 min | §3.2 |
| 3 | Notación Big-O — definición formal | 20 min | §3.3 |
| 4 | Big-Ω, Big-Θ y relaciones entre ellas | 10 min | §3.3 |
| 5 | Reglas prácticas para análisis de código | 15 min | §3.3 |
| 6 | Estudios de caso: `prefix_average`, `disjoint` | 20 min | §3.3 |
| 🧪 | Tests y autoevaluación | — | — |

> **Nota pedagógica:** Las secciones 3–5 son el núcleo. Si el tiempo escasea, priorizalas.

## ⚙️ Configuración del Entorno

In [ ]:
import sys
import math
import time
import timeit
import random
import unittest
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

assert sys.version_info >= (3, 8), "Se requiere Python 3.8+"
print(f"Python {sys.version}")
print("Módulos cargados: math, time, timeit, random, unittest, matplotlib, numpy")

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print("Listo para el análisis de algoritmos 📐")

---

## §3.1 — ¿Por qué analizar algoritmos?

**Problema central:** dado un problema, existen múltiples algoritmos que lo resuelven. ¿Cómo elegir el mejor?

### Enfoque experimental (ingenuo)

La idea más directa es medir el tiempo de ejecución real:

```python
inicio = time.perf_counter()
resultado = mi_algoritmo(datos)
fin = time.perf_counter()
print(f"Tiempo: {fin - inicio:.6f} segundos")
```

### ⚠️ Limitaciones del enfoque experimental

| Limitación | Explicación |
|-----------|-------------|
| **Hardware dependiente** | Una CPU más rápida da mejores tiempos, no un mejor algoritmo |
| **Requiere implementación** | Hay que codificar antes de comparar |
| **Entradas limitadas** | Solo se mide para los casos de prueba elegidos |
| **Factores externos** | SO, caché, carga del sistema afectan la medición |

### Enfoque matemático (asintótico)

Caracterizamos el comportamiento como función del **tamaño de la entrada** `n`, independientemente del hardware.

> **Intuición:** En vez de preguntar *"¿cuántos segundos tarda?"*, preguntamos *"¿cuántas operaciones primitivas realiza en función de n?"*

In [ ]:
# Comparación experimental: búsqueda lineal vs binaria
def busqueda_lineal(lista, objetivo):
    """O(n): recorre hasta encontrar o agotar la lista."""
    for i, elem in enumerate(lista):
        if elem == objetivo:
            return i
    return -1

def busqueda_binaria(lista_ordenada, objetivo):
    """O(log n): divide el espacio de búsqueda a la mitad cada vez."""
    izq, der = 0, len(lista_ordenada) - 1
    while izq <= der:
        mid = (izq + der) // 2
        if lista_ordenada[mid] == objetivo:
            return mid
        elif lista_ordenada[mid] < objetivo:
            izq = mid + 1
        else:
            der = mid - 1
    return -1

print(f"{'n':>10} {'Lineal (µs)':>14} {'Binaria (µs)':>14}")
print("-" * 42)
for n in [1_000, 10_000, 100_000, 1_000_000]:
    datos = list(range(n))
    objetivo = n - 1
    t_lin = timeit.timeit(lambda: busqueda_lineal(datos, objetivo), number=100) / 100 * 1e6
    t_bin = timeit.timeit(lambda: busqueda_binaria(datos, objetivo), number=1000) / 1000 * 1e6
    print(f"{n:>10,} {t_lin:>14.1f} {t_bin:>14.3f}")

print("\n⚠️  Los tiempos cambian con el hardware.")
print("La relación RELATIVA (lineal vs binaria) es lo estable.")

### Observación

En la tabla anterior, ambas funciones crecen con `n`. Pero la lineal crece proporcionalmente a `n` mientras que la binaria crece con **log(n)** — mucho más despacio.

Los tiempos absolutos variarán según la máquina, pero la **razón** entre ellos permanece similar. Esa razón es lo que captura la notación asintótica.

---

## §3.2 — Las Siete Funciones de Crecimiento

El libro identifica **siete funciones** que aparecen repetidamente en el análisis de algoritmos. Ordenadas de menor a mayor crecimiento:

| Clase | Notación | Ejemplo típico |
|-------|---------|----------------|
| Constante | $O(1)$ | Acceso a lista por índice, push a pila |
| Logarítmica | $O(\log n)$ | Búsqueda binaria, altura de árbol balanceado |
| Lineal | $O(n)$ | Búsqueda lineal, `max()`, recorrido completo |
| N-log-N | $O(n \log n)$ | Merge sort, Heap sort |
| Cuadrática | $O(n^2)$ | Burbuja, selección, doble bucle simple |
| Cúbica | $O(n^3)$ | Triple bucle, matrices naive |
| Exponencial | $O(2^n)$ | Subconjuntos, fuerza bruta combinatoria |

### La jerarquía fundamental

$$1 \;\ll\; \log n \;\ll\; n \;\ll\; n \log n \;\ll\; n^2 \;\ll\; n^3 \;\ll\; 2^n$$

> **Regla de oro:** Si el algoritmo es $O(n^2)$ y la entrada tiene $n = 10^6$, el resultado es $10^{12}$ operaciones — inviable. Con $O(n \log n)$ son solo $\approx 2 \times 10^7$.

### Comparación numérica

| n | O(log n) | O(n) | O(n log n) | O(n²) | O(2ⁿ) |
|---|----------|------|------------|-------|-------|
| 10 | 3 | 10 | 33 | 100 | 1,024 |
| 100 | 7 | 100 | 664 | 10,000 | ~10³⁰ |
| 1,000 | 10 | 1,000 | 9,966 | 1,000,000 | ∞ |

In [ ]:
# Visualización de las funciones de crecimiento
n = np.linspace(1, 20, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

funciones = {
    'O(1)':        (np.ones_like(n),       'gray',   '-'),
    'O(log n)':    (np.log2(n),            'blue',   '--'),
    'O(n)':        (n,                     'green',  '-'),
    'O(n log n)':  (n * np.log2(n),        'orange', '--'),
    'O(n²)':       (n**2,                  'red',    '-'),
    'O(n³)':       (n**3,                  'purple', ':'),
}

for label, (y, color, ls) in funciones.items():
    ax1.plot(n, y, label=label, color=color, linestyle=ls, linewidth=2)
ax1.set_xlim(1, 20); ax1.set_ylim(0, 400)
ax1.set_xlabel('n (tamaño de entrada)'); ax1.set_ylabel('Operaciones')
ax1.set_title('Clases de complejidad (escala lineal)')
ax1.legend(loc='upper left', fontsize=9)

for label, (y, color, ls) in funciones.items():
    ax2.plot(n[1:], y[1:], label=label, color=color, linestyle=ls, linewidth=2)
ax2.set_yscale('log'); ax2.set_xlabel('n')
ax2.set_ylabel('Operaciones (escala log)')
ax2.set_title('Clases de complejidad (escala log)')
ax2.legend(loc='upper left', fontsize=9)

plt.suptitle('Las Siete Funciones de Crecimiento — Cap. 3 §3.2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## §3.3 — Notación Asintótica

### Big-O: cota superior asintótica

> **Definición:** $f(n)$ es $O(g(n))$ si existen constantes $c > 0$ y $n_0 \geq 1$ tales que:
> $$f(n) \leq c \cdot g(n) \quad \forall\, n \geq n_0$$

**Interpretación:** para entradas suficientemente grandes, $f(n)$ no crece más rápido que $g(n)$ (salvo un factor constante).

### Reglas de simplificación

| Regla | Antes | Después |
|-------|-------|---------|
| Descartar constantes | $3n^2$ | $O(n^2)$ |
| Descartar términos menores | $5n^2 + 3n + 7$ | $O(n^2)$ |
| Suma: domina el mayor | $O(n^2) + O(n)$ | $O(n^2)$ |
| Producto: multiplicar | $O(n) \cdot O(\log n)$ | $O(n \log n)$ |

### Big-Ω y Big-Θ

| Notación | Significado | Interpretación |
|----------|------------|----------------|
| $O(g(n))$ | Cota **superior** | "No crece más rápido que $g$" |
| $\Omega(g(n))$ | Cota **inferior** | "No crece más lento que $g$" |
| $\Theta(g(n))$ | Cota **ajustada** | Es tanto $O$ como $\Omega$ |

En la práctica, cuando se dice que un algoritmo es $O(n^2)$, casi siempre se quiere decir $\Theta(n^2)$.

In [ ]:
# Ejemplos de análisis Big-O del libro §3.3
def find_max(data):
    """Máximo de una secuencia — O(n)."""
    biggest = data[0]
    for val in data:
        if val > biggest:
            biggest = val
    return biggest


def prefix_average1(S):
    """Promedios prefijos — O(n^2). A[j] = media(S[0..j])."""
    n = len(S)
    A = [0] * n
    for j in range(n):
        total = 0
        for i in range(j + 1):
            total += S[i]
        A[j] = total / (j + 1)
    return A


def prefix_average2(S):
    """Promedios prefijos — O(n). Suma acumulada evita recomputar."""
    n = len(S)
    A = [0] * n
    total = 0
    for j in range(n):
        total += S[j]
        A[j] = total / (j + 1)
    return A


S = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]
a1 = prefix_average1(S)
a2 = prefix_average2(S)
print(f"S            = {S}")
print(f"prefix_avg1  = {[round(x, 2) for x in a1]}")
print(f"prefix_avg2  = {[round(x, 2) for x in a2]}")
print(f"Iguales:       {a1 == a2}")

ops = sum(range(1, len(S) + 1))
print(f"\nOperaciones prefix_average1(n={len(S)}): {ops}")
print(f"Fórmula n(n+1)/2 = {len(S)*(len(S)+1)//2}")
print("-> O(n²) confirmado")

In [ ]:
# Benchmark: prefix_average1 O(n²) vs prefix_average2 O(n)
tamanios = [100, 500, 1_000, 3_000, 5_000]
t1_list, t2_list = [], []

print(f"{'n':>7} {'O(n²) ms':>12} {'O(n) ms':>10} {'speedup':>9}")
print("-" * 44)
for n in tamanios:
    S = list(range(n))
    t1 = timeit.timeit(lambda: prefix_average1(S), number=20) / 20 * 1000
    t2 = timeit.timeit(lambda: prefix_average2(S), number=200) / 200 * 1000
    t1_list.append(t1); t2_list.append(t2)
    ratio = t1 / max(t2, 1e-9)
    print(f"{n:>7,} {t1:>12.3f} {t2:>10.4f} {ratio:>8.1f}x")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tamanios, t1_list, 'r-o', label='prefix_avg1  O(n²)', linewidth=2)
axes[0].plot(tamanios, t2_list, 'g-o', label='prefix_avg2  O(n)',  linewidth=2)
axes[0].set_xlabel('n'); axes[0].set_ylabel('Tiempo (ms)')
axes[0].set_title('Escala lineal'); axes[0].legend()

axes[1].loglog(tamanios, t1_list, 'r-o', label='prefix_avg1  O(n²)', linewidth=2)
axes[1].loglog(tamanios, t2_list, 'g-o', label='prefix_avg2  O(n)',  linewidth=2)
axes[1].set_xlabel('n'); axes[1].set_ylabel('Tiempo (ms)')
axes[1].set_title('Escala log-log (pendiente = orden)'); axes[1].legend()

plt.suptitle('prefix_average: O(n²) vs O(n)', fontweight='bold')
plt.tight_layout(); plt.show()

---

## §3.3 — Reglas prácticas para analizar código

### 1. Bucle simple
```python
for i in range(n):       # n iteraciones
    operacion_O1()       # O(1) cada una
# -> O(n) total
```

### 2. Bucles anidados dependientes
```python
for i in range(n):
    for j in range(i):   # varía 0..n-1
        operacion_O1()
# sum(0..n-1) = n(n-1)/2 -> O(n²)
```

### 3. Secuencia de pasos
```python
paso_A(n)   # O(n)
paso_B(n)   # O(n log n)
# -> O(n log n): domina el mayor
```

### 4. Condicional
```python
if condicion:
    bloque_O_n()    # O(n)
else:
    bloque_O_1()    # O(1)
# -> O(n): peor caso
```

### 5. Loop de duplicación
```python
i = 1
while i < n:
    i *= 2     # i: 1, 2, 4, ..., n -> log₂(n) pasos
# -> O(log n)
```

In [ ]:
# Análisis de complejidad paso a paso
def example_a(n):
    """Bucle simple -> O(n)."""
    total = 0
    for i in range(n):
        total += i
    return total

def example_b(n):
    """Bucles anidados dependientes -> O(n²)."""
    total = 0
    for i in range(n):
        for j in range(i):
            total += 1
    return total

def example_c(n):
    """Loop de duplicación -> O(log n)."""
    i = n
    while i > 1:
        i //= 2
    return i

print("Iteraciones de example_b (O(n²)):")
for n in [5, 10, 20, 50]:
    c = sum(range(n))
    print(f"  n={n:>3}: {c:>5} ops  (n(n-1)/2 = {n*(n-1)//2})")

print("\nPasos de example_c (O(log n)):")
for n in [2, 8, 64, 1024]:
    steps = 0; i = n
    while i > 1:
        i //= 2; steps += 1
    print(f"  n={n:>5}: {steps:>3} pasos  (log₂ = {math.log2(n):.1f})")

---

## 🧪 Tests

In [ ]:
class TestFindMax(unittest.TestCase):
    def test_lista_normal(self):
        self.assertEqual(find_max([3, 1, 4, 1, 5, 9, 2, 6]), 9)
    def test_lista_un_elemento(self):
        self.assertEqual(find_max([42]), 42)
    def test_negativos(self):
        self.assertEqual(find_max([-5, -1, -3]), -1)

class TestPrefixAverage(unittest.TestCase):
    def test_ambos_dan_mismo_resultado(self):
        S = [1, 2, 3, 4, 5]
        self.assertEqual(prefix_average1(S), prefix_average2(S))
    def test_primer_elemento_igual(self):
        S = [10, 20, 30]
        a = prefix_average2(S)
        self.assertEqual(a[0], 10.0)
    def test_promedio_conocido(self):
        S = [2, 4, 6]
        a = prefix_average2(S)
        self.assertAlmostEqual(a[2], 4.0)

res = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestSuite([
        unittest.TestLoader().loadTestsFromTestCase(TestFindMax),
        unittest.TestLoader().loadTestsFromTestCase(TestPrefixAverage),
    ]))
print(f"\n{'OK' if res.wasSuccessful() else 'FAILED'} — {res.testsRun} tests")

In [ ]:
# three_way_disjoint (§3.3.3)
def disjoint1(A, B, C):
    """O(n³): triple bucle anidado."""
    for a in A:
        for b in B:
            for c in C:
                if a == b == c:
                    return False
    return True

def disjoint2(A, B, C):
    """O(n²): el bucle sobre C solo corre cuando a==b."""
    for a in A:
        for b in B:
            if a == b:
                for c in C:
                    if a == c:
                        return False
    return True

A = [1, 2, 3]; B = [4, 5, 6]; C = [7, 8, 9]
print(f"disjoint1 (disjuntos): {disjoint1(A,B,C)}")
print(f"disjoint2 (disjuntos): {disjoint2(A,B,C)}")

A2 = [1, 2, 3]; B2 = [2, 4, 5]; C2 = [6, 7, 2]
print(f"disjoint1 (con intersección): {disjoint1(A2,B2,C2)}")
print(f"disjoint2 (con intersección): {disjoint2(A2,B2,C2)}")

In [ ]:
# Benchmark disjoint1 O(n³) vs disjoint2 O(n²)
print(f"{'n':>6} {'disjoint1 (ms)':>16} {'disjoint2 (ms)':>16} {'speedup':>9}")
print("-" * 52)
for n in [50, 100, 200, 400]:
    A = list(range(n)); B = list(range(n, 2*n)); C = list(range(2*n, 3*n))
    t1 = timeit.timeit(lambda: disjoint1(A,B,C), number=5)/5*1000
    t2 = timeit.timeit(lambda: disjoint2(A,B,C), number=50)/50*1000
    speedup = t1/max(t2, 1e-9)
    print(f"{n:>6} {t1:>16.2f} {t2:>16.3f} {speedup:>8.1f}x")

---

## 📚 Resumen — Capítulo 3

### Las clases de complejidad en orden

$$O(1) \subset O(\log n) \subset O(n) \subset O(n \log n) \subset O(n^2) \subset O(n^3) \subset O(2^n)$$

### Tabla de referencia rápida

| Complejidad | Nombre | Ejemplo típico | ¿Viable para n=10⁶? |
|------------|--------|----------------|----------------------|
| O(1) | Constante | Acceso por índice, push | ✅ Siempre |
| O(log n) | Logarítmica | Búsqueda binaria | ✅ Siempre |
| O(n) | Lineal | Recorrido, `find_max` | ✅ Sí |
| O(n log n) | Cuasi-lineal | Merge sort | ✅ Sí (~2×10⁷) |
| O(n²) | Cuadrática | Burbuja, `prefix_average1` | ⚠️ n≤10⁴ |
| O(n³) | Cúbica | `disjoint1` | ❌ n≤10³ |
| O(2ⁿ) | Exponencial | Subconjuntos | ❌ n≤30 |

### Autoevaluación

- [ ] Puedo ordenar las 7 funciones sin mirar la tabla
- [ ] Dado código con bucles, determino su complejidad
- [ ] Entiendo por qué `prefix_average2` es más eficiente que `prefix_average1`
- [ ] Puedo explicar la diferencia entre O, Ω y Θ con un ejemplo
- [ ] Sé por qué `disjoint2` es O(n²) aunque tiene tres bucles

## 📖 Referencias y Conexiones

### Fuentes principales
- **Goodrich, Tamassia & Goldwasser** — *Data Structures and Algorithms in Python*, Cap. 3
  - §3.1: Experimental studies
  - §3.2: The seven functions
  - §3.3: Asymptotic analysis — Big-O, Ω, Θ

### Conexiones con otros capítulos

| Capítulo | Por qué importa el análisis |
|----------|------------------------------|
| Cap. 5 — Arrays | `append()` amortizado O(1) vs O(n) peor caso |
| Cap. 6 — Stacks/Queues | `deque.appendleft` O(1) vs `list.insert(0,x)` O(n) |
| Cap. 7 — Listas enlazadas | Inserción O(1) al frente vs O(n) para índice |
| Cap. 9 — Colas de prioridad | Heap: insert O(log n), extract-min O(log n) |
| Cap. 12 — Sorting | Merge sort O(n log n), bubble sort O(n²) |

---

**Notebook 1 — Notación Asintótica y Funciones de Crecimiento**  
*Estructuras de Datos y Algoritmos en Python — Cap. 3*  
Goodrich, Tamassia & Goldwasser

Material didáctico bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).